In [1]:
# Imports
#import keypoint_moseq as kpms
import os
import h5py
import numpy as np
from scipy.interpolate import interp1d

In [3]:
# Interpolation
def fill_missing(Y, kind="linear"):
    """Fills missing values independently along each dimension after the first."""
    initial_shape = Y.shape
    Y = Y.reshape((initial_shape[0], -1))

    for i in range(Y.shape[-1]):
        y = Y[:, i]
        x = np.flatnonzero(~np.isnan(y))
        if len(x) < 2:
            continue  # Skip if not enough points to interpolate

        f = interp1d(x, y[x], kind=kind, fill_value=np.nan, bounds_error=False)
        xq = np.flatnonzero(np.isnan(y))
        y[xq] = f(xq)

        mask = np.isnan(y)
        if np.any(~mask):
            y[mask] = np.interp(np.flatnonzero(mask), np.flatnonzero(~mask), y[~mask])

        Y[:, i] = y

    return Y.reshape(initial_shape)

In [ ]:
#interpolate for a single h5 file
filename = r"\\smith-nas.ucsd.edu\lab\2024-09_FoodDep_Amanda\SLEAP\female\h5 (by only)\20240913_FoodDep_F_postCAP_2092_2093_bottom.analysis.h5"

with h5py.File(filename, "r") as f:
    dset_names = list(f.keys())
    locations = f["tracks"][:].T
    node_names = [n.decode() for n in f["node_names"][:]]

print("===filename===")
print(filename)
print()

print("===HDF5 datasets===")
print(dset_names)
print()

print("===locations data shape===")
print(locations.shape)
print()

print("===nodes===")
for i, name in enumerate(node_names):
    print(f"{i}: {name}")
print()
locations = fill_missing(locations)
print(np.isnan(locations).any())

===filename===
\\smith-nas.ucsd.edu\lab\2024-09_FoodDep_Amanda\SLEAP\female\h5 (by only)\20240913_FoodDep_F_postCAP_2092_2093_bottom.analysis.h5

===HDF5 datasets===
['edge_inds', 'edge_names', 'instance_scores', 'labels_path', 'node_names', 'point_scores', 'provenance', 'track_names', 'track_occupancy', 'tracking_scores', 'tracks', 'video_ind', 'video_path']

===locations data shape===
(54000, 8, 2, 1)

===nodes===
0: nose
1: R_frontpaw
2: L_frontpaw
3: R_hindpaw
4: L_hindpaw
5: tail_base
6: tail_end
7: body_center



In [5]:
# Apply interpolation code to each h5 file
input_folder = r"Z:\2024-09_FoodDep_Amanda\SLEAP\female\h5 (by and cap)"
output_folder = r"Z:\2024-09_FoodDep_Amanda\SLEAP\female\h5 (by and cap)\interpolated"
os.makedirs(output_folder, exist_ok=True)

for filename in os.listdir(input_folder):
    if filename.endswith(".h5"):
        filepath = os.path.join(input_folder, filename)
        output_path = os.path.join(output_folder, filename)
        
        with h5py.File(filepath, "r") as f:
            dset_names = list(f.keys())
            locations = f["tracks"][:].T
            node_names = [n.decode() for n in f["node_names"][:]]

            print(f"Processing file: {filepath}")
            print(f"Datasets: {dset_names}")
            print(f"Locations shape: {locations.shape}")
            print(f"Nodes: {node_names}")

            # Interpolate
            data_interp = fill_missing(locations)

            # Save interpolated version
            with h5py.File(output_path, "w") as f_out:
                # Copy all datasets and attributes
                for key in f.keys():
                    if key == "tracks":
                        # Replace the "tracks" dataset with interpolated data
                        f_out.create_dataset("tracks", data=data_interp.T)
                    else:
                        # Copy other datasets as is
                        f.copy(key, f_out)
                # Copy attributes from the root level
                for attr_name, attr_value in f.attrs.items():
                    f_out.attrs[attr_name] = attr_value
        print(np.isnan(data_interp).any())
        print(f"Interpolated and saved: {output_path}")

Processing file: Z:\2024-09_FoodDep_Amanda\SLEAP\female\h5 (by and cap)\20240919_FoodDep_F_postCAP_2108_2109_bottom.analysis.h5
Datasets: ['edge_inds', 'edge_names', 'instance_scores', 'labels_path', 'node_names', 'point_scores', 'provenance', 'track_names', 'track_occupancy', 'tracking_scores', 'tracks', 'video_ind', 'video_path']
Locations shape: (54000, 8, 2, 2)
Nodes: ['nose', 'R_frontpaw', 'L_frontpaw', 'R_hindpaw', 'L_hindpaw', 'tail_base', 'tail_end', 'body_center']
False
Interpolated and saved: Z:\2024-09_FoodDep_Amanda\SLEAP\female\h5 (by and cap)\interpolated\20240919_FoodDep_F_postCAP_2108_2109_bottom.analysis.h5
Processing file: Z:\2024-09_FoodDep_Amanda\SLEAP\female\h5 (by and cap)\20240919_FoodDep_F_postCAP_2112_2113_bottom.analysis.h5
Datasets: ['edge_inds', 'edge_names', 'instance_scores', 'labels_path', 'node_names', 'point_scores', 'provenance', 'track_names', 'track_occupancy', 'tracking_scores', 'tracks', 'video_ind', 'video_path']
Locations shape: (54000, 8, 2, 2)
